<div class="alert alert-block alert-info">
<b>Welcome to SynEdu!</b> This talktorial is part of <b>SynEdu</b>, a lightweight teaching series built around the <b>Syn</b> ecosystem and <b>RDKit</b> for practical, reproducible cheminformatics.
</div>

<div class="alert alert-block alert-warning">
<b>Reproducibility first</b>: Keep runtime short, prefer small datasets, and pin dependencies (e.g., via <code>env/environment.yml</code>). Save version info alongside exported figures.
</div>

<div class="alert alert-block alert-success">
<b>By the end of this notebook</b>, you will be able to <b>validate atom-mapped reactions</b> and catch common issues before extracting graph rewrite rules.
</div>

# S01 · Atom-mapping sanity checks (RDKit)

**Data:** `data/reactions_mapped.csv`


## Authors and contributions

- Tieu-Long Phan, Peter Stadler group, Professur für Bioinformatik, Institut für Informatik, Universität Leipzig
- (Add contributors here)


<div class="alert alert-block alert-info">
<b>Cross-referencing</b>: When referring to another SynEdu notebook, use <b>Talktorial SXX</b> (e.g., <b>Talktorial S03</b>).
</div>


## Roadmap
- Concepts: what atom mapping encodes, why 1–1 mapping matters
- Hands-on: parse mapped SMILES, compute QC signals, visualize map numbers


# Theory

Atom mapping attaches an integer label (atom-map number) to atoms so that
reactant atoms can be traced to product atoms.

Typical QC checks:
- duplicate map numbers on the same side
- missing map numbers across sides
- unmapped atoms (map number = 0)


# Practical


In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import networkx as nx

from rdkit import Chem
from rdkit.Chem import Draw

# Optional: Syn ecosystem (kept optional for Paper 1)
try:
    import synkit  # type: ignore
    HAS_SYNKit = True
except Exception:
    HAS_SYNKit = False

OUT = Path("talktorials/out")
OUT.mkdir(parents=True, exist_ok=True)

import rdkit
import networkx as nx_mod
print("RDKit:", rdkit.__version__)
print("NetworkX:", nx_mod.__version__)
print("SynKit available:", HAS_SYNKit)


In [ ]:
from collections import Counter

df = pd.read_csv("data/reactions_mapped.csv")
df


In [ ]:
def atom_map_counter(smiles: str) -> Counter:
    m = Chem.MolFromSmiles(smiles)
    c = Counter()
    if m is None:
        return c
    for a in m.GetAtoms():
        amap = a.GetAtomMapNum()
        if amap:
            c[amap] += 1
    return c

def mapping_qc(am_rxn_smiles: str) -> dict:
    react, prod = am_rxn_smiles.split(">>")
    R = atom_map_counter(react)
    P = atom_map_counter(prod)
    return {
        "n_unique_R": len(R),
        "n_unique_P": len(P),
        "dup_R": sorted([k for k,v in R.items() if v > 1]),
        "dup_P": sorted([k for k,v in P.items() if v > 1]),
        "missing_in_P": sorted(set(R) - set(P)),
        "missing_in_R": sorted(set(P) - set(R)),
    }

qc_rows = []
for _, r in df.iterrows():
    qc_rows.append({"rxn_id": r.rxn_id, "label": r.label, **mapping_qc(r.am_rxn_smiles)})
qc_df = pd.DataFrame(qc_rows)
qc_df


In [ ]:
# Visualize one mapped reaction: show map numbers on atoms
row = df.iloc[0]
react, prod = row.am_rxn_smiles.split(">>")

def draw_mapnums(smiles: str, size=(450,250)):
    m = Chem.MolFromSmiles(smiles)
    for a in m.GetAtoms():
        amap = a.GetAtomMapNum()
        if amap:
            a.SetProp("atomNote", str(amap))
    return Draw.MolToImage(m, size=size)

display(draw_mapnums(react))
display(draw_mapnums(prod))


# Discussion

- Mapping QC is the cheapest place to fail fast: bad mappings propagate into bad rules.
- Some reactions legitimately create/delete atoms (e.g., leaving groups); missing maps can be valid if you treat spectators carefully.


# Quiz
1. What does a duplicate map number in reactants indicate?
2. When can a map number appear only on the product side?
3. Extend QC to count how many atoms are unmapped (map=0).


# References and further reading

*Suggested citation style:*  
* Keyword: <i>Source</i> (year) (link)

- RDKit documentation: <i>RDKit</i> (ongoing) — https://www.rdkit.org/docs/
- RDKit Book: <i>The RDKit Book</i> (ongoing) — https://www.rdkit.org/docs/Book.html
- NetworkX documentation: <i>NetworkX</i> (ongoing) — https://networkx.org/documentation/stable/
- Graphviz DOT language: <i>Graphviz</i> (ongoing) — https://graphviz.org/documentation/
